# 音声認識や OCR の誤り率 (WER / CER) を計算する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/13_evaluation/wer_cer.ipynb)

NLP クックブック 13-03 ／ 評価指標と統計的検定

音声認識や OCR の出力が、正解とどれくらい違うかを単語誤り率・文字誤り率で測りたい。

In [1]:
!pip install -q jiwer

In [2]:
import jiwer

ref = ["the quick brown fox jumps over the lazy dog", "natural language processing is fun"]
hyp = ["the quick brown fox jumped over lazy dog", "natural language processing fun is"]
out = jiwer.process_words(ref, hyp)
print(f"WER = {out.wer:.3f}  (置換 {out.substitutions} / 削除 {out.deletions} / 挿入 {out.insertions} / 正解 {out.hits})")
print(jiwer.visualize_alignment(out))

WER = 0.286  (置換 1 / 削除 2 / 挿入 1 / 正解 11)
=== SENTENCE 1 ===

REF: the quick brown fox  jumps over the lazy dog
HYP: the quick brown fox jumped over *** lazy dog
                              S        D         

=== SENTENCE 2 ===

REF: natural language processing *** is fun
HYP: natural language processing fun is ***
                                   I      D

=== SUMMARY ===
number of sentences: 2
substitutions=1 deletions=2 insertions=1 hits=11

mer=26.67%
wil=33.52%
wip=66.48%
wer=28.57%



In [3]:
# 日本語は単語境界がないので、文字誤り率 (CER) を使うのが一般的
import unicodedata

def normalize(s):
    s = unicodedata.normalize("NFKC", s)
    return "".join(c for c in s if not unicodedata.category(c).startswith(("P", "Z")))   # 句読点と空白を除く

ref_ja = ["今日は良い天気ですね。", "自然言語処理の研究をしています。"]
hyp_ja = ["今日はいい天気ですね", "自然原語処理の研究をしてます。"]
print("CER (そのまま):", round(jiwer.cer(ref_ja, hyp_ja), 3))
print("CER (正規化後):", round(jiwer.cer([normalize(s) for s in ref_ja], [normalize(s) for s in hyp_ja]), 3))

CER (そのまま): 0.148
CER (正規化後): 0.12


- WER =（置換 + 削除 + 挿入）÷ 参照の単語数。挿入が多いと 100% を超えることもある。
- 「良い / いい」「3 / 三」のような表記の違いも誤りに数えられる。評価前の正規化（句読点、全角半角、数字の表記）の方針を決めて明記する。